# 03d Assemble Modeling Feature Table

This notebook reads the Step 03c cleaned country-month-year outputs and writes one wide modeling feature table plus validation reports.


In [ ]:
from pathlib import Path  # Work with filesystem paths.

REPO_URL = "https://github.com/Arobnett/HDX-sources-and-more-API-connection.git"  # Store the GitHub repository URL.
REPO_DIR = Path("/content/HDX-sources-and-more-API-connection")  # Define the local Colab clone folder.

if not REPO_DIR.exists():  # Clone the repo only when this runtime has not cloned it yet.
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/HDX-sources-and-more-API-connection
!git pull  # Refresh the Colab clone after GitHub edits.


In [ ]:
!git lfs install  # Enable Git LFS support in this Colab runtime.
!git lfs pull  # Download real large source files instead of pointer files.


In [ ]:
import sys  # Access Python's module search path.
from pathlib import Path  # Represent repository paths independently of operating system.

PROJECT_ROOT = Path.cwd()  # Treat the cloned repository as the project root.
SRC_DIR = PROJECT_ROOT / "src"  # Locate shared Python modules.
sys.path.insert(0, str(SRC_DIR))  # Make src modules importable in this Colab session.

from feature_assembly import assemble_feature_table  # Import the Step 03d feature assembly runner.
from paths import CLEAN_DIR, FEATURE_REPORTS_DIR, MODEL_FEATURES_DIR, ensure_output_directories  # Import canonical paths.

print(f"Project root: {PROJECT_ROOT}")  # Show the active repository root.
print(f"Cleaned feature directory exists: {CLEAN_DIR.exists()}")  # Confirm Step 03c outputs are available.
print(f"Cleaned feature files: {len(list(CLEAN_DIR.glob('*.csv')))}")  # Count cleaned source files.


In [ ]:
ensure_output_directories()  # Create generated output folders only.

assembled_features, assembly_report, feature_catalog = assemble_feature_table()  # Assemble all cleaned sources into one wide table.

assembly_report  # Display source-level assembly validation.


In [ ]:
print(f"Assembled rows: {len(assembled_features)}")  # Print final row count.
print(f"Assembled columns: {len(assembled_features.columns)}")  # Print final column count.
print(f"Feature columns: {len(assembled_features.columns) - 4}")  # Print non-key feature count.

display(assembled_features.head())  # Preview the first assembled rows.
display(feature_catalog.head(20))  # Preview the source-to-feature mapping.


In [ ]:
source_key_errors = assembly_report[assembly_report["key_unique"].eq(False)]  # Identify cleaned source tables with duplicate keys.
assembled_key_errors = assembly_report[assembly_report["assembled_key_unique"].eq(False)]  # Identify final assembled duplicate-key failures.
missing_feature_catalog = feature_catalog[feature_catalog["assembled_column"].isna()] if "assembled_column" in feature_catalog.columns else feature_catalog  # Guard against catalog schema failures.

print(f"Source tables with duplicate country-month keys: {len(source_key_errors)}")  # Print source key issue count.
print(f"Assembled duplicate-key failures: {len(assembled_key_errors)}")  # Print final key issue count.
print(f"Catalog rows missing assembled column names: {len(missing_feature_catalog)}")  # Print catalog issue count.

if len(source_key_errors) or len(assembled_key_errors) or len(missing_feature_catalog):  # Stop when assembly failed validation.
    raise ValueError("03d feature assembly validation failed; inspect assembly_report and feature_catalog.")  # Raise a visible notebook error.

print("03d smoke test passed: assembled feature table and reports were written.")  # Confirm successful run.


In [ ]:
print(f"Feature table: {MODEL_FEATURES_DIR / 'model_feature_table_country_month_year.csv'}")  # Show assembled table path.
print(f"Assembly report: {FEATURE_REPORTS_DIR / 'feature_assembly_report.csv'}")  # Show assembly report path.
print(f"Feature catalog: {FEATURE_REPORTS_DIR / 'feature_column_catalog.csv'}")  # Show feature catalog path.
